## Tratamiento de Imagenes Dedos

## Instalación

```bash
pip install --upgrade numpy scipy pandas matplotlib pillow opencv-python scikit-learn seaborn
```


## Importaciones

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

## Configuración general

In [5]:
RUTA_DATA = Path("dedo")
RANDOM_STATE = 42
TEST_SIZE = 0.25
PANEL_SIZE = (640, 480)
TARGET_BACKGROUND_GRAY = 200
TARGET_BOX_GRAY = 25
DARK_BOX_THRESHOLD = 25
HAND_BACKGROUND_THRESHOLD = 120
HAND_LIGHT_THRESHOLD = 200
LIGHT_SATURATION_THRESHOLD = 120

## Cargar el dataset `dedo`

No vamos a usar una selección manual de ejemplos. Cogemos todas las imágenes de la carpeta `dedo`.

In [6]:
registros = []

for etiqueta_dir in sorted(RUTA_DATA.iterdir()):
    if not etiqueta_dir.is_dir():
        continue
    for ruta in sorted(etiqueta_dir.glob("*.png")):
        registros.append(
            {
                "ruta": str(ruta),
                "etiqueta": etiqueta_dir.name,
                "archivo": ruta.name,
            }
        )

datos = pd.DataFrame(registros)
datos.head()

,ruta,etiqueta,archivo
0,dedo/no/img_1776863541097.png,no,img_1776863541097.png
1,dedo/no/img_1776863844568.png,no,img_1776863844568.png
2,dedo/no/img_1776863884377.png,no,img_1776863884377.png
3,dedo/no/img_1776863887385.png,no,img_1776863887385.png
4,dedo/no/img_1776863902961.png,no,img_1776863902961.png


In [7]:
datos.groupby("etiqueta").size().rename("n").reset_index()

,etiqueta,n
0,no,37
1,si,44


## Separación train/test con semilla fija

Esta es la única "subdivisión" que haremos: reservar un conjunto de test reproducible.

In [8]:
train_df, test_df = train_test_split(
    datos,
    test_size=TEST_SIZE,
    stratify=datos["etiqueta"],
    random_state=RANDOM_STATE,
)

train_df = train_df.copy()
train_df["split"] = "train"

test_df = test_df.copy()
test_df["split"] = "test"

dataset_df = pd.concat([train_df, test_df], ignore_index=True)
dataset_df.head()

,ruta,etiqueta,archivo,split
0,dedo/no/img_1776863887385.png,no,img_1776863887385.png,train
1,dedo/no/img_1777028310895.png,no,img_1777028310895.png,train
2,dedo/si/img_1777027592482.png,si,img_1777027592482.png,train
3,dedo/si/img_1776864612420.png,si,img_1776864612420.png,train
4,dedo/si/img_1777028190117.png,si,img_1777028190117.png,train


In [9]:
pd.crosstab(dataset_df["split"], dataset_df["etiqueta"], margins=True)

etiqueta,no,si,All
split,,,
test,10,11,21
train,27,33,60
All,37,44,81
